# Fine-tuning de video preentrenado para BARD

Este cuaderno implementa una pipeline end-to-end en PyTorch para distinguir entre **Shot** y **Pass** en BARD.

La idea central es mantener el flujo modular para poder cambiar el backbone entre **X3D** y **VideoMAE** con un solo parámetro, y usar un esquema de validación por `val_loss` para seleccionar el mejor checkpoint.

## Requisitos y estructura esperada

1. Descarga vídeos con `BARD/src/download_video.py` (o colócalos en `BARD/data/`).
2. Prepara el dataset ejecutando:

```bash
python prepare_bard_dataset.py
```

Eso genera:

```text
data/
  catalog.csv
  summary.json
  splits/
    train.csv
    val.csv
    test.csv
  train/
    pass/
    shot/
  val/
    pass/
    shot/
  test/
    pass/
    shot/
```

El cuaderno entrena directamente sobre esa carpeta `data/`.

Si usas `VideoMAE`, instala `transformers`. Si usas `X3D`, instala `pytorchvideo`. El cuaderno usa además `torchvision`, `scikit-learn`, `pandas` y `matplotlib`.

In [5]:
# Si estás en un entorno limpio, descomenta lo siguiente en Jupyter.
# %pip install torch torchvision pytorchvideo scikit-learn pandas matplotlib
# %pip install transformers

import av
import os
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from torchvision.transforms import CenterCrop, Compose, Normalize, Resize


def read_video_pyav(path: str, pts_unit: str = 'sec'):
    """Load video as [T, H, W, C] uint8 tensor (PyAV; torchvision 0.27+ removed read_video)."""
    del pts_unit  # kept for API compatibility with the old torchvision call
    container = av.open(path)
    frames = []
    try:
        for frame in container.decode(video=0):
            frames.append(frame.to_ndarray(format='rgb24'))
    finally:
        container.close()

    if not frames:
        raise RuntimeError(f'El video está vacío o no se pudo decodificar: {path}')

    video = torch.from_numpy(np.stack(frames))
    return video, None, {}

try:
    from pytorchvideo.transforms import UniformTemporalSubsample
except ImportError:
    # Fallback ligero para que el cuaderno siga siendo ejecutable si PyTorchVideo no está instalado.
    class UniformTemporalSubsample:
        def __init__(self, num_samples: int):
            self.num_samples = num_samples

        def __call__(self, video: torch.Tensor) -> torch.Tensor:
            if video.ndim != 4:
                raise ValueError(f'Expected video tensor [C, T, H, W], got {tuple(video.shape)}')
            total_frames = video.shape[1]
            if total_frames == 0:
                raise ValueError('Empty video tensor received.')
            if total_frames == self.num_samples:
                return video
            indices = torch.linspace(0, total_frames - 1, self.num_samples).long()
            return video[:, indices, :, :]

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from transformers import VideoMAEForVideoClassification
except ImportError:
    VideoMAEForVideoClassification = None

torch.backends.cudnn.benchmark = torch.cuda.is_available()

def set_seed(seed: int) -> None:
    # Mantener la semilla fija ayuda a comparar mejor experimentos low-dat a.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## Dataloader

Lee los splits ya preparados en `data/` (`train/`, `val/`, `test/` con subcarpetas `pass/` y `shot/`). Cada vídeo se convierte a un clip fijo de `NUM_FRAMES` fotogramas para compatibilidad con X3D y VideoMAE.

In [ ]:
@dataclass
class Config:
    data_root: Path = Path('./data')
    output_dir: Path = Path('./artifacts')
    backbone: str = 'x3d'  # Cambia a 'videomae' si quieres probar el transformer.
    num_frames: int = 16
    resize_size: int = 256
    crop_size: int = 224
    batch_size: int = 2
    num_workers: int = 0 if os.name == 'nt' else min(4, os.cpu_count() or 1)
    epochs: int = 10
    lr: float = 1e-5
    weight_decay: float = 1e-4
    seed: int = 42
    use_pretrained: bool = True

cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)
set_seed(cfg.seed)

if not cfg.data_root.exists():
    raise FileNotFoundError(
        f'No existe {cfg.data_root}. Ejecuta primero: python prepare_bard_dataset.py'
    )

KINETICS_MEAN = [0.43216, 0.394666, 0.37645]
KINETICS_STD = [0.22803, 0.22145, 0.216989]
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

def load_dataset_summary(data_root: Path) -> dict:
    summary_path = data_root / 'summary.json'
    if not summary_path.exists():
        raise FileNotFoundError(
            f'No existe {summary_path}. Ejecuta primero: python prepare_bard_dataset.py'
        )
    with open(summary_path, encoding='utf-8') as f:
        return json.load(f)

def discover_classes(data_root: Path, summary: dict) -> List[str]:
    if 'class_names' in summary:
        return summary['class_names']
    train_dir = data_root / 'train'
    classes = sorted(item.name for item in train_dir.iterdir() if item.is_dir())
    if not classes:
        raise ValueError(f'No se encontraron clases dentro de {train_dir}.')
    return classes

def scan_split_directory(split_dir: Path, class_to_idx: Dict[str, int]) -> List[Tuple[Path, int]]:
    samples: List[Tuple[Path, int]] = []
    for class_name, label in class_to_idx.items():
        class_dir = split_dir / class_name
        if not class_dir.exists():
            continue
        for video_path in sorted(class_dir.rglob('*')):
            if video_path.is_file() and video_path.suffix.lower() in VIDEO_EXTENSIONS:
                samples.append((video_path, label))
    if not samples:
        raise ValueError(f'No se encontraron videos válidos en {split_dir}.')
    return samples

class BardVideoDataset(Dataset):
    def __init__(self, samples: List[Tuple[Path, int]], num_frames: int, resize_size: int, crop_size: int):
        self.samples = samples
        self.temporal = UniformTemporalSubsample(num_frames)
        self.spatial = Compose([
            Resize(resize_size),
            CenterCrop(crop_size),
            Normalize(mean=KINETICS_MEAN, std=KINETICS_STD),
        ])

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        video_path, label = self.samples[idx]
        video, _, _ = read_video_pyav(str(video_path), pts_unit='sec')
        if video.numel() == 0:
            raise RuntimeError(f'El video está vacío: {video_path}')

        video = video.permute(3, 0, 1, 2).float() / 255.0
        video = self.temporal(video)
        frames = [self.spatial(frame) for frame in video.unbind(dim=1)]
        video = torch.stack(frames, dim=1)
        return video, torch.tensor(label, dtype=torch.long), idx, str(video_path)

def build_dataloaders(cfg: Config):
    summary = load_dataset_summary(cfg.data_root)
    class_names = discover_classes(cfg.data_root, summary)
    class_to_idx = summary.get('class_to_idx', {name: idx for idx, name in enumerate(class_names)})

    train_samples = scan_split_directory(cfg.data_root / 'train', class_to_idx)
    val_samples = scan_split_directory(cfg.data_root / 'val', class_to_idx)
    test_samples = scan_split_directory(cfg.data_root / 'test', class_to_idx)

    train_dataset = BardVideoDataset(train_samples, cfg.num_frames, cfg.resize_size, cfg.crop_size)
    val_dataset = BardVideoDataset(val_samples, cfg.num_frames, cfg.resize_size, cfg.crop_size)
    test_dataset = BardVideoDataset(test_samples, cfg.num_frames, cfg.resize_size, cfg.crop_size)

    pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=pin_memory)

    print('Clases:', class_names)
    print('Resumen del dataset:', summary.get('counts', {}))
    print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

    return train_loader, val_loader, test_loader, class_names, class_to_idx

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Modelo

El cuaderno permite cambiar entre dos familias de backbone sin tocar el resto del pipeline:

- **X3D**: opción práctica si quieres una arquitectura eficiente en vídeo con soporte directo de PyTorchVideo.
- **VideoMAE**: útil si prefieres un transformer preentrenado fuerte sobre Kinetics.

La capa final se reemplaza por dos clases. Para `X3D`, se reescribe la cabeza de clasificación. Para `VideoMAE`, se carga el checkpoint con `num_labels=2` y `ignore_mismatched_sizes=True`, lo que fuerza la adaptación de la capa final.

In [ ]:
# El modelo activo está definido en la celda siguiente como `VideoClassifier`.
# Soporta backbones `x3d` (torchvision) y `videomae` (transformers).

## Training loop, validación y análisis de error

Se usa `CrossEntropyLoss`, `AdamW` con learning rate pequeño y guardado del mejor checkpoint según `val_loss`.

La función de evaluación devuelve además los índices y rutas de los vídeos mal clasificados. Eso es importante para el análisis de fallo porque te permite revisar manualmente qué patrones están confundiendo al modelo.

In [ ]:
def plot_confusion_matrix(cm: np.ndarray, class_names: List[str], path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    plt.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()

def build_misclassified_df(metrics: Dict, idx_to_class: Dict[int, str]) -> pd.DataFrame:
    rows = []
    for i, (true_label, pred_label) in enumerate(zip(metrics['y_true'], metrics['y_pred'])):
        if true_label == pred_label:
            continue
        rows.append({
            'index': metrics['indices'][i],
            'path': metrics['paths'][i],
            'true_label': idx_to_class[true_label],
            'pred_label': idx_to_class[pred_label],
        })
    return pd.DataFrame(rows)

def replace_last_linear_layer(module: nn.Module, num_classes: int) -> nn.Module:
    # Reemplazar la última capa lineal es una forma robusta de adaptar modelos preentrenados con distintas cabezas.
    linear_modules = [(name, child) for name, child in module.named_modules() if isinstance(child, nn.Linear)]
    if not linear_modules:
        raise ValueError('No se encontró ninguna capa Linear para reemplazar.')

    linear_path, linear_layer = linear_modules[-1]
    parent = module
    path_parts = linear_path.split('.')
    for part in path_parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, path_parts[-1], nn.Linear(linear_layer.in_features, num_classes))
    return module

class VideoClassifier(nn.Module):
    def __init__(self, backbone_name: str, num_classes: int = 2, use_pretrained: bool = True):
        super().__init__()
        self.backbone_name = backbone_name.lower()

        if self.backbone_name == 'x3d':
            from torchvision.models.video import X3D_M_Weights, x3d_m
            weights = X3D_M_Weights.DEFAULT if use_pretrained else None
            self.backbone = x3d_m(weights=weights)
            self.backbone = replace_last_linear_layer(self.backbone, num_classes)

        elif self.backbone_name == 'videomae':
            if VideoMAEForVideoClassification is None:
                raise ImportError('Instala transformers para usar VideoMAE.')
            checkpoint = 'MCG-NJU/videomae-base-finetuned-kinetics'
            self.backbone = VideoMAEForVideoClassification.from_pretrained(
                checkpoint,
                num_labels=num_classes,
                ignore_mismatched_sizes=True,
            )
        else:
            raise ValueError(
                f'Backbone no soportado: {backbone_name}. Usa "x3d" o "videomae".'
            )

    def forward(self, videos: torch.Tensor) -> torch.Tensor:
        if self.backbone_name == 'videomae':
            # VideoMAE suele esperar [B, T, C, H, W], por eso reordenamos el tensor aquí.
            videos = videos.permute(0, 2, 1, 3, 4)
            outputs = self.backbone(pixel_values=videos)
            return outputs.logits
        return self.backbone(videos)

def build_optimizer(model: nn.Module, cfg: Config):
    # AdamW es una elección estándar para fine-tuning de Transformers y CNNs profundas.
    return torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

def save_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, best_val_loss: float, class_names: List[str], cfg: Config) -> None:
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_loss': best_val_loss,
        'class_names': class_names,
        'config': cfg.__dict__,
        'backbone': cfg.backbone,
    }, path)

def load_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer | None = None, map_location: str | torch.device = 'cpu'):
    checkpoint = torch.load(path, map_location=map_location)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint

def train_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer, device: torch.device) -> float:
    model.train()
    running_loss = 0.0
    total_samples = 0

    for videos, labels, _, _ in loader:
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(videos)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

    return running_loss / max(1, total_samples)

def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device, return_details: bool = False):
    model.eval()
    running_loss = 0.0
    total_samples = 0
    y_true: List[int] = []
    y_pred: List[int] = []
    indices: List[int] = []
    paths: List[str] = []

    with torch.no_grad():
        for videos, labels, batch_indices, batch_paths in loader:
            videos = videos.to(device)
            labels = labels.to(device)
            logits = model(videos)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = logits.argmax(dim=1)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

            if return_details:
                indices.extend(batch_indices.tolist())
                paths.extend(list(batch_paths))

    metrics = {
        'loss': running_loss / max(1, total_samples),
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'y_true': y_true,
        'y_pred': y_pred,
    }

    if return_details:
        metrics['indices'] = indices
        metrics['paths'] = paths

    return metrics

def fit_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer, device: torch.device, cfg: Config, class_names: List[str]):
    history = []
    best_val_loss = float('inf')
    best_checkpoint_path = cfg.output_dir / 'best_checkpoint.pt'

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate(model, val_loader, criterion, device, return_details=False)

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_f1_macro': val_metrics['f1_macro'],
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | val_f1={val_metrics['f1_macro']:.4f}"
        )

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            save_checkpoint(best_checkpoint_path, model, optimizer, epoch, best_val_loss, class_names, cfg)
            print(f"  -> Nuevo mejor checkpoint guardado en {best_checkpoint_path}")

    history_df = pd.DataFrame(history)
    return history_df, best_checkpoint_path

## Ejecución end-to-end

La siguiente celda carga los splits, construye el backbone, entrena con validación y luego evalúa sobre test.

Si quieres cambiar el modelo base, modifica `cfg.backbone` en la celda de configuración. Si quieres usar VideoMAE, instala `transformers` y cambia esa misma opción a `videomae`.

In [ ]:
train_loader, val_loader, test_loader, class_names, class_to_idx = build_dataloaders(cfg)
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}
num_classes = len(class_names)

model = VideoClassifier(cfg.backbone, num_classes=num_classes, use_pretrained=cfg.use_pretrained).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = build_optimizer(model, cfg)

history_df, best_checkpoint_path = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    cfg=cfg,
    class_names=class_names,
)

best_checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.to(device)

test_metrics = evaluate(model, test_loader, criterion, device, return_details=True)
misclassified_df = build_misclassified_df(test_metrics, idx_to_class)
cm = confusion_matrix(test_metrics['y_true'], test_metrics['y_pred'], labels=list(range(num_classes)))
report = classification_report(test_metrics['y_true'], test_metrics['y_pred'], target_names=class_names)

print('\n--- Resultado en test ---')
print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test F1 macro: {test_metrics['f1_macro']:.4f}")
print('\nClassification report:')
print(report)

history_path = cfg.output_dir / 'training_history.csv'
history_df.to_csv(history_path, index=False)

errors_path = cfg.output_dir / 'misclassified_test_samples.csv'
misclassified_df.to_csv(errors_path, index=False)

cm_path = cfg.output_dir / 'confusion_matrix_test.png'
plot_confusion_matrix(cm, class_names, cm_path)

print(f"Historial guardado en: {history_path}")
print(f"Errores guardados en: {errors_path}")
print(f"Matriz de confusión guardada en: {cm_path}")

display(misclassified_df.head(20))

if not history_df.empty:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history_df['epoch'], history_df['train_loss'], label='train loss')
    ax[0].plot(history_df['epoch'], history_df['val_loss'], label='val loss')
    ax[0].set_title('Loss')
    ax[0].legend()

    ax[1].plot(history_df['epoch'], history_df['val_accuracy'], label='val acc')
    ax[1].set_title('Validation accuracy')
    ax[1].legend()

    plt.tight_layout()
    plt.show()